# Plan B (First-Observation) — RAD-DINO fine-tune trên study CXR đầu tiên / bệnh nhân

**Khác bản gốc ở cohort:** mỗi bệnh nhân chỉ lấy **study CXR sớm nhất** (theo StudyDate+StudyTime) và **giữ MỌI view** trong study đó (thường 1–3 ảnh). Gắn thêm **anchor EHR**: lần nhập viện có ICD-10 **đầu tiên** (`first_hadm_id`, `first_admittime`, `first_icd10_codes`) để align cho fusion sau.

Phần cache / fine-tune / eval / trích embedding giống bản gốc. Output ghi vào `planB_firstobs_out/`.

## 1. Môi trường

In [ ]:
!pip install -q transformers timm huggingface_hub pyarrow scikit-image seaborn

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
# from huggingface_hub import login; login()

## 2. Mount + cấu hình

In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()

import os
BASE        = '/content/drive/MyDrive/MIMIC_CXR/mimic-cxr-jpg-2.1.0.physionet.org'
PLANA_DIR   = '/content/drive/MyDrive/MIMIC_CXR/planA_out'
OUT_DIR     = '/content/drive/MyDrive/MIMIC_CXR/planB_firstobs_out'
CACHE_DIR   = os.path.join(OUT_DIR, 'cache')
LOCAL_CACHE = '/content/planB_firstobs_cache'
for d in (OUT_DIR, CACHE_DIR, LOCAL_CACHE): os.makedirs(d, exist_ok=True)

BILLING_PROJECT = 'mimic-ehr-v310'
REPO            = 'microsoft/rad-dino'

CACHE_RES, TRAIN_RES, CACHE_SHARD = 256, 224, 4000
N_UNFREEZE, EPOCHS, BATCH = 3, 12, 64
LR_BACKBONE, LR_HEAD, WEIGHT_DECAY = 1e-5, 1e-3, 0.05

CHEXPERT_LABELS = [
 'Atelectasis','Cardiomegaly','Consolidation','Edema','Enlarged Cardiomediastinum',
 'Fracture','Lung Lesion','Lung Opacity','No Finding','Pleural Effusion',
 'Pleural Other','Pneumonia','Pneumothorax','Support Devices']
Y_COLS = ['y_'+c for c in CHEXPERT_LABELS]

# ==== Cohort theo TÊN CHẨN ĐOÁN (long_title) — gồm cả ICD-9 + ICD-10, lấy TỐI ĐA ca ====
H = "physionet-data.mimiciv_3_1_hosp"
I = "physionet-data.mimiciv_3_1_icu"
# Điều kiện long_title cho từng nhóm; {t} = LOWER(dd.long_title). CHỈNH TẠI ĐÂY.
DISEASE_TITLE = {
  'pneumonia':      "{t} LIKE '%pneumonia%'",
  'bronchitis':     "{t} LIKE '%bronchitis%'",
  'tuberculosis':   "{t} LIKE '%tuberculosis%'",
  'arthritis':      "{t} LIKE '%arthritis%'",
  'fracture':       "{t} LIKE '%fracture%'",
  'scoliosis':      "{t} LIKE '%scoliosis%'",
  'lung_cancer':    "({t} LIKE '%malignant%' AND {t} LIKE '%lung%')",
  'lung_infection': "(({t} LIKE '%lung%' OR {t} LIKE '%pulmonary%') AND {t} LIKE '%infection%')",
}
def group_cond(g):
    return "(" + DISEASE_TITLE[g].format(t="LOWER(dd.long_title)") + ")"
_ALL = " OR ".join(group_cond(g) for g in DISEASE_TITLE)
TARGET_ADM = (f"SELECT DISTINCT d.subject_id, d.hadm_id "
              f"FROM `{H}.diagnoses_icd` d JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version) "
              f"WHERE {_ALL}")
print("OK. out:", OUT_DIR)

## 3. Cohort + anchor EHR (lần nhập viện ICD-10 đầu tiên)

In [ ]:
import pandas as pd, numpy as np, os
from google.cloud import bigquery

manifest = pd.read_parquet(os.path.join(PLANA_DIR, 'manifest.parquet'))
print("Manifest Plan A:", len(manifest), "ảnh |", manifest.subject_id.nunique(), "bệnh nhân")

client = bigquery.Client(project=BILLING_PROJECT)
H = "physionet-data.mimiciv_3_1_hosp"
ehr = client.query(f"""
  WITH icd10_adm AS ({TARGET_ADM}),
  first_adm AS (
    SELECT a.subject_id, a.hadm_id, a.admittime,
           ROW_NUMBER() OVER (PARTITION BY a.subject_id ORDER BY a.admittime, a.hadm_id) AS rn
    FROM `{H}.admissions` a
    JOIN icd10_adm i USING (subject_id, hadm_id)
  )
  SELECT f.subject_id,
         f.hadm_id   AS first_hadm_id,
         f.admittime AS first_admittime,
         STRING_AGG(d.icd_code, '|' ORDER BY d.seq_num) AS first_icd10_codes
  FROM first_adm f
  LEFT JOIN `{H}.diagnoses_icd` d
    ON f.subject_id = d.subject_id AND f.hadm_id = d.hadm_id AND d.icd_version = 10
  WHERE f.rn = 1
  GROUP BY f.subject_id, f.hadm_id, f.admittime
""").to_dataframe()
print("Bệnh nhân có lần nhập viện ICD-10:", len(ehr))

cohort_ids = set(manifest.subject_id.unique()) & set(ehr.subject_id.tolist())
ehr = ehr[ehr.subject_id.isin(cohort_ids)].reset_index(drop=True)
print("Giao CXR ∩ ICD-10:", len(cohort_ids), "bệnh nhân")

cohort_df = pd.DataFrame({'subject_id': sorted(cohort_ids)})
cohort_df.to_parquet(os.path.join(OUT_DIR,'cohort_subjects.parquet'), index=False)
cohort_df.to_csv(os.path.join(OUT_DIR,'cohort_subjects.csv'), index=False)
print("Đã lưu cohort_subjects.csv:", len(cohort_df), "bệnh nhân")

## 4. Study CXR sớm nhất/bệnh nhân (mọi view) + nhãn U-Ones + dataset.csv

In [ ]:
# StudyDate/StudyTime từ metadata để xếp thời gian
meta = pd.read_csv(os.path.join(BASE,'mimic-cxr-2.0.0-metadata.csv.gz'),
                   usecols=['dicom_id','StudyDate','StudyTime'])
cm = manifest[manifest.subject_id.isin(cohort_ids)].merge(meta, on='dicom_id', how='left')

# study sớm nhất mỗi bệnh nhân
st = (cm.groupby(['subject_id','study_id'])
        .agg(sd=('StudyDate','min'), stime=('StudyTime','min')).reset_index()
        .sort_values(['subject_id','sd','stime']))
first_study = st.groupby('subject_id', as_index=False).first()[['subject_id','study_id']]

# giữ TẤT CẢ ảnh (mọi view) của study sớm nhất
cm = cm.merge(first_study, on=['subject_id','study_id'], how='inner').reset_index(drop=True)
# gắn anchor EHR
cm = cm.merge(ehr, on='subject_id', how='left')

# resolve nhãn: blank(NaN)->0 ; uncertain(-1)->1 (U-Ones)
L = cm[CHEXPERT_LABELS].to_numpy('float32')
Y = np.where(np.isnan(L), 0.0, np.where(L==-1.0, 1.0, L)).astype('float32')
for j,c in enumerate(CHEXPERT_LABELS): cm['y_'+c] = Y[:,j]

cm.to_parquet(os.path.join(OUT_DIR,'manifest_cohort.parquet'), index=False)
cm.to_csv(os.path.join(OUT_DIR,'dataset.csv'), index=False)
print("Dataset:", len(cm), "ảnh /", cm.subject_id.nunique(), "bệnh nhân (study đầu, mọi view)")
print("Ảnh/bệnh nhân trung bình:", round(len(cm)/cm.subject_id.nunique(),2))
print(cm.view.value_counts())
print(cm.split.value_counts(dropna=False))
tr = cm[cm.split=='train']
print("Prevalence train:")
for c in CHEXPERT_LABELS:
    print(f"  {c:<28} pos={int(tr['y_'+c].sum()):>6} ({100*tr['y_'+c].mean():.1f}%)")

## 5. Trích embedding FROZEN 768-dim từ RAD-DINO (đọc GCS)\n\nRAD-DINO pretrained trên MIMIC → dùng **frozen** (không fine-tune). Đọc ảnh song song từ GCS, forward, lưu shard resumable.

In [ ]:
from google.colab import auth; auth.authenticate_user()
from google.cloud import storage
import google.auth
from google.auth.transport.requests import AuthorizedSession
from requests.adapters import HTTPAdapter
from concurrent.futures import ThreadPoolExecutor
import io, os, glob, time, numpy as np, pandas as pd, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import AutoModel, AutoImageProcessor

BUCKET='mimic-cxr-jpg-2.1.0.physionet.org'; THREADS=32
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=AutoModel.from_pretrained(REPO).eval().to(device)
for p in model.parameters(): p.requires_grad=False
proc=AutoImageProcessor.from_pretrained(REPO)
tf=T.Compose([T.Resize(TRAIN_RES),T.CenterCrop(TRAIN_RES),T.ToTensor(),T.Normalize(proc.image_mean,proc.image_std)])
cm=pd.read_parquet(os.path.join(OUT_DIR,'manifest_cohort.parquet'))
cm['rel']=cm['jpg_path'].str.split('mimic-cxr-jpg-2.1.0.physionet.org/').str[-1]
EMB_C=[f'cls_{i}' for i in range(768)]; EMB_M=[f'mean_{i}' for i in range(768)]
SHARD_DIR=os.path.join(OUT_DIR,'emb_shards'); os.makedirs(SHARD_DIR,exist_ok=True)
_bucket=None
def _get_bucket():
    global _bucket
    if _bucket is None:
        creds,_=google.auth.default(); sess=AuthorizedSession(creds)
        sess.mount('https://', HTTPAdapter(pool_connections=THREADS,pool_maxsize=THREADS))
        _bucket=storage.Client(project=BILLING_PROJECT, credentials=creds, _http=sess).bucket(BUCKET, user_project=BILLING_PROJECT)
    return _bucket
def _one(rel):
    try: return tf(Image.open(io.BytesIO(_get_bucket().blob(rel).download_as_bytes())).convert('RGB'))
    except Exception: return None
class BatchGcsDS(Dataset):
    def __init__(self, df, bs): self.rel=df['rel'].tolist(); self.id=df['dicom_id'].tolist(); self.bs=bs
    def __len__(self): return (len(self.id)+self.bs-1)//self.bs
    def __getitem__(self, b):
        lo=b*self.bs; hi=min(lo+self.bs,len(self.id)); ids=self.id[lo:hi]; rels=self.rel[lo:hi]
        with ThreadPoolExecutor(THREADS) as ex: ims=list(ex.map(_one, rels))
        keep=[(i,im) for i,im in zip(ids,ims) if im is not None]
        if not keep: return [], None
        return [k[0] for k in keep], torch.stack([k[1] for k in keep])
print("OK. cohort ảnh:", len(cm), "| device:", device)

In [ ]:
from tqdm.auto import tqdm
done=set()
for fpar in glob.glob(os.path.join(SHARD_DIR,'*.parquet')):
    done |= set(pd.read_parquet(fpar, columns=['dicom_id']).dicom_id.tolist())
todo = cm[~cm.dicom_id.isin(done)].reset_index(drop=True)
print("Đã trích:", len(done), "| còn:", len(todo))
loader = DataLoader(BatchGcsDS(todo, BATCH), batch_size=None, num_workers=4, prefetch_factor=2)
sh = len(glob.glob(os.path.join(SHARD_DIR,'*.parquet')))
buf_id, buf_c, buf_m, seen, t0 = [],[],[],0,time.time()
SAVE_EVERY=4000
def flush():
    global sh, buf_id, buf_c, buf_m
    dfp=pd.concat([pd.DataFrame(np.vstack(buf_c),columns=EMB_C),pd.DataFrame(np.vstack(buf_m),columns=EMB_M)],axis=1)
    dfp.insert(0,'dicom_id',buf_id); dfp.to_parquet(os.path.join(SHARD_DIR,f'emb_{sh:05d}.parquet'),index=False); sh+=1
    buf_id, buf_c, buf_m = [],[],[]
pbar=tqdm(loader, total=len(loader))
for ids, x in pbar:
    if x is None: continue
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        h=model(pixel_values=x.to(device), interpolate_pos_encoding=True).last_hidden_state
        c=h[:,0].float().cpu().numpy(); m=h[:,1:].mean(1).float().cpu().numpy()
    buf_id+=ids; buf_c.append(c); buf_m.append(m); seen+=len(ids)
    pbar.set_postfix(img_s=round(seen/(time.time()-t0),1), done=seen)
    if sum(a.shape[0] for a in buf_c) >= SAVE_EVERY: flush()
if buf_id: flush()
print("XONG:", seen, "| shard:", sh)

In [ ]:
files=sorted(glob.glob(os.path.join(SHARD_DIR,'*.parquet')))
alldf=pd.concat([pd.read_parquet(f) for f in files], ignore_index=True).drop_duplicates('dicom_id')
idx=cm[cm.dicom_id.isin(set(alldf.dicom_id))].reset_index(drop=True)
alldf=alldf.set_index('dicom_id').loc[idx.dicom_id].reset_index()
np.save(os.path.join(OUT_DIR,'embeddings_raddino_ft_cls.npy'), alldf[EMB_C].to_numpy('float32'))
np.save(os.path.join(OUT_DIR,'embeddings_raddino_ft_meanpool.npy'), alldf[EMB_M].to_numpy('float32'))
out=idx[['dicom_id','subject_id','study_id','split','view']].copy(); out['row_index']=np.arange(len(out))
out.to_parquet(os.path.join(OUT_DIR,'embeddings_raddino_index.parquet'), index=False)
print("Frozen embeddings:", len(alldf), "ảnh × 768 (CLS + mean-pool)")

## 6. Đánh giá 14 nhãn — linear probe (class_weight balanced + tune ngưỡng)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
emb=np.load(os.path.join(OUT_DIR,'embeddings_raddino_ft_cls.npy'))
idx=pd.read_parquet(os.path.join(OUT_DIR,'embeddings_raddino_index.parquet'))
cmm=pd.read_parquet(os.path.join(OUT_DIR,'manifest_cohort.parquet'))
d=idx.merge(cmm[['dicom_id']+['y_'+c for c in CHEXPERT_LABELS]], on='dicom_id', how='left')
tr,va,te=(d.split=='train').values,(d.split=='validate').values,(d.split=='test').values
sc=StandardScaler().fit(emb[tr]); Xtr,Xva,Xte=sc.transform(emb[tr]),sc.transform(emb[va]),sc.transform(emb[te])
rows=[]
for c in CHEXPERT_LABELS:
    ytr,yva,yte=d.loc[tr,'y_'+c].values,d.loc[va,'y_'+c].values,d.loc[te,'y_'+c].values
    if len(np.unique(ytr))<2: rows.append((c,np.nan,np.nan,np.nan,np.nan)); continue
    clf=LogisticRegression(max_iter=2000, class_weight='balanced').fit(Xtr,ytr)
    pva,pte=clf.predict_proba(Xva)[:,1],clf.predict_proba(Xte)[:,1]
    ths=np.linspace(0.05,0.95,19)
    bt=max(ths,key=lambda t: f1_score(yva,pva>=t,zero_division=0)) if len(np.unique(yva))==2 else 0.5
    pred=pte>=bt; au=roc_auc_score(yte,pte) if len(np.unique(yte))==2 else np.nan
    rows.append((c,au,precision_score(yte,pred,zero_division=0),recall_score(yte,pred,zero_division=0),f1_score(yte,pred,zero_division=0)))
met=pd.DataFrame(rows,columns=['label','AUROC','precision','recall','F1'])
met.to_csv(os.path.join(OUT_DIR,'raddino_frozen_linearprobe.csv'),index=False)
print(met.to_string(index=False)); print("Macro AUROC:",round(met.AUROC.mean(),4),"| Macro F1:",round(met.F1.mean(),4))

## 7. Feature EHR per-CXR (MODE: encounter / firstadm) — inpatient + ED

In [ ]:
import pandas as pd, numpy as np, os
from google.cloud import bigquery
client = bigquery.Client(project=BILLING_PROJECT)
H="physionet-data.mimiciv_3_1_hosp"; I="physionet-data.mimiciv_3_1_icu"; ED="physionet-data.mimiciv_ed"

MODE = 'encounter'   # 'encounter' = EHR theo lần khám chứa mỗi CXR ; 'firstadm' = lần nhập viện target đầu

cm  = pd.read_parquet(os.path.join(OUT_DIR,'manifest_cohort.parquet'))
COH = set(cm.subject_id.unique())

if MODE == 'encounter':
    meta = pd.read_csv(os.path.join(BASE,'mimic-cxr-2.0.0-metadata.csv.gz'),
                       usecols=['dicom_id','subject_id','study_id','StudyDate','StudyTime'])
    meta = meta[meta.dicom_id.isin(set(cm.dicom_id))].copy()
    _dd=meta.StudyDate.astype('int64').astype(str).str.zfill(8)
    _tt=meta.StudyTime.fillna(0).astype(float).astype('int64').astype(str).str.zfill(6).str.slice(0,6)
    meta['study_dt']=pd.to_datetime(_dd+_tt, format='%Y%m%d%H%M%S', errors='coerce')
    stt = meta[['dicom_id','subject_id','study_id','study_dt']].dropna(subset=['study_dt']).reset_index(drop=True)
    adm = client.query(f"SELECT subject_id,hadm_id,admittime,dischtime FROM `{H}.admissions`").to_dataframe()
    adm = adm[adm.subject_id.isin(COH)]; adm['admittime']=pd.to_datetime(adm.admittime); adm['dischtime']=pd.to_datetime(adm.dischtime)
    eds = client.query(f"SELECT subject_id,stay_id,hadm_id,intime,outtime FROM `{ED}.edstays`").to_dataframe()
    eds = eds[eds.subject_id.isin(COH)]; eds['intime']=pd.to_datetime(eds.intime); eds['outtime']=pd.to_datetime(eds.outtime)
    ji = stt.merge(adm,on='subject_id'); ji = ji[(ji.study_dt>=ji.admittime)&(ji.study_dt<=ji.dischtime)]
    ji = ji.sort_values('admittime').drop_duplicates('dicom_id')[['dicom_id','hadm_id']]
    je = stt.merge(eds,on='subject_id'); je = je[(je.study_dt>=je.intime)&(je.study_dt<=je.outtime)]
    je = je.sort_values('intime').drop_duplicates('dicom_id')[['dicom_id','stay_id','hadm_id']].rename(columns={'hadm_id':'ed_hadm'})
    link = stt[['dicom_id','subject_id','study_id']].merge(ji,on='dicom_id',how='left').merge(je,on='dicom_id',how='left')
    link['hadm_id'] = link['hadm_id'].fillna(link['ed_hadm'])
else:
    fa = client.query(f"""
      WITH ta AS ({TARGET_ADM}),
      fadm AS (SELECT a.subject_id,a.hadm_id, ROW_NUMBER() OVER (PARTITION BY a.subject_id ORDER BY a.admittime,a.hadm_id) rn
               FROM `{H}.admissions` a JOIN ta USING(subject_id,hadm_id))
      SELECT subject_id,hadm_id FROM fadm WHERE rn=1""").to_dataframe()
    link = cm[['dicom_id','subject_id','study_id']].drop_duplicates('dicom_id').merge(fa,on='subject_id',how='left')
    link['stay_id'] = np.nan
print(f"[{MODE}] CXR {len(link)} | có hadm {link.hadm_id.notna().sum()} | có ED stay {int(link.stay_id.notna().sum())} | không link {(link.hadm_id.isna()&link.stay_id.isna()).sum()}")

CO = f"WITH cohort AS (SELECT DISTINCT subject_id FROM ({TARGET_ADM}))"
demo = client.query(f"{CO} SELECT p.subject_id,p.anchor_age age,p.gender FROM `{H}.patients` p JOIN cohort USING(subject_id)").to_dataframe()
admattr = client.query(f"{CO} SELECT a.hadm_id,a.admission_type,a.insurance,a.race,a.marital_status FROM `{H}.admissions` a JOIN cohort c ON c.subject_id=a.subject_id").to_dataframe()
labs = client.query(f"""{CO}
SELECT le.hadm_id,
 AVG(IF(dl.label='Hemoglobin',le.valuenum,NULL)) lab_hemoglobin, AVG(IF(dl.label='White Blood Cells',le.valuenum,NULL)) lab_wbc,
 AVG(IF(dl.label='Red Blood Cells',le.valuenum,NULL)) lab_rbc, AVG(IF(dl.label='Platelet Count',le.valuenum,NULL)) lab_platelet,
 AVG(IF(dl.label='Creatinine',le.valuenum,NULL)) lab_creatinine, AVG(IF(dl.label='Sodium',le.valuenum,NULL)) lab_sodium,
 AVG(IF(dl.label='Potassium',le.valuenum,NULL)) lab_potassium, AVG(IF(dl.label='Glucose',le.valuenum,NULL)) lab_glucose,
 AVG(IF(dl.label='Urea Nitrogen',le.valuenum,NULL)) lab_bun, AVG(IF(dl.label='Chloride',le.valuenum,NULL)) lab_chloride
FROM `{H}.labevents` le JOIN cohort c ON c.subject_id=le.subject_id JOIN `{H}.d_labitems` dl ON dl.itemid=le.itemid
WHERE le.valuenum IS NOT NULL AND le.hadm_id IS NOT NULL GROUP BY le.hadm_id""").to_dataframe()
vicu = client.query(f"""{CO}
SELECT ce.hadm_id, AVG(IF(di.label='Heart Rate',ce.valuenum,NULL)) hr_icu, AVG(IF(di.label='O2 saturation pulseoxymetry',ce.valuenum,NULL)) spo2_icu,
 AVG(CASE WHEN di.label='Temperature Celsius' THEN ce.valuenum WHEN di.label='Temperature Fahrenheit' THEN (ce.valuenum-32)*5/9 END) temp_icu
FROM `{I}.chartevents` ce JOIN cohort c ON c.subject_id=ce.subject_id JOIN `{I}.d_items` di ON di.itemid=ce.itemid
WHERE ce.valuenum IS NOT NULL AND ce.hadm_id IS NOT NULL GROUP BY ce.hadm_id""").to_dataframe()
tri = client.query(f"""{CO}
SELECT t.stay_id, t.heartrate hr_ed, t.o2sat spo2_ed, t.resprate vital_resprate, t.temperature temp_f, t.sbp vital_sbp, t.dbp vital_dbp, t.chiefcomplaint
FROM `{ED}.triage` t JOIN cohort c ON c.subject_id=t.subject_id""").to_dataframe()
tri['temp_ed'] = (tri['temp_f']-32)*5/9

f = (link.merge(demo,on='subject_id',how='left').merge(admattr,on='hadm_id',how='left')
         .merge(labs,on='hadm_id',how='left').merge(vicu,on='hadm_id',how='left')
         .merge(tri[['stay_id','hr_ed','spo2_ed','temp_ed','vital_resprate','vital_sbp','vital_dbp','chiefcomplaint']],on='stay_id',how='left'))
f['vital_heart_rate']=f['hr_icu'].fillna(f['hr_ed']); f['vital_spo2']=f['spo2_icu'].fillna(f['spo2_ed']); f['vital_temperature']=f['temp_icu'].fillna(f['temp_ed'])
f['ed_cough']=f['chiefcomplaint'].fillna('').str.contains('cough',case=False).astype(int)
assert (f.subject_id.isin(COH)).all()

micro = client.query(f"{CO} SELECT m.subject_id, m.spec_type_desc specimen, m.org_name FROM `{H}.microbiologyevents` m JOIN cohort c ON c.subject_id=m.subject_id").to_dataframe()
def onehot(df,col,k,pfx):
    s=df.dropna(subset=[col])
    if len(s)==0: return pd.DataFrame({'subject_id':[]})
    top=s[col].value_counts().head(k).index; s=s[s[col].isin(top)]
    oh=(pd.crosstab(s.subject_id,s[col])>0).astype(int); oh.columns=[pfx+str(c).strip().replace(' ','_').replace('/','_') for c in oh.columns]
    return oh.reset_index()
org_oh=onehot(micro,'org_name',20,'org_'); spec_oh=onehot(micro,'specimen',15,'spec_')
has_pos=micro.dropna(subset=['org_name']).groupby('subject_id').size().rename('n_positive_cultures').reset_index()
flags = client.query(f"""{CO}
SELECT d.subject_id,
 MAX(CASE WHEN (d.icd_version=10 AND (d.icd_code LIKE 'F17%' OR d.icd_code IN('Z720','Z87891'))) OR (d.icd_version=9 AND d.icd_code IN('3051','V1582')) THEN 1 ELSE 0 END) smoke,
 MAX(CASE WHEN (d.icd_version=10 AND d.icd_code LIKE 'R05%') OR (d.icd_version=9 AND d.icd_code='7862') THEN 1 ELSE 0 END) cough
FROM `{H}.diagnoses_icd` d JOIN cohort c ON c.subject_id=d.subject_id GROUP BY d.subject_id""").to_dataframe()
for t_ in (org_oh,spec_oh,has_pos,flags): f=f.merge(t_,on='subject_id',how='left')

cols=['dicom_id','subject_id','study_id','hadm_id','stay_id','age','gender','admission_type','insurance','race','marital_status',
 'lab_hemoglobin','lab_wbc','lab_rbc','lab_platelet','lab_creatinine','lab_sodium','lab_potassium','lab_glucose','lab_bun','lab_chloride',
 'vital_heart_rate','vital_spo2','vital_temperature','vital_resprate','vital_sbp','vital_dbp','chiefcomplaint','ed_cough',
 'smoke','cough','n_positive_cultures']+[c for c in f.columns if c.startswith('org_') or c.startswith('spec_')]
ehr = f[[c for c in cols if c in f.columns]].copy()
numc=[c for c in ehr.columns if c.startswith('lab_') or c.startswith('vital_') or c.startswith('org_') or c.startswith('spec_') or c in ('age','n_positive_cultures','smoke','cough','ed_cough')]
for c in numc: ehr[c]=pd.to_numeric(ehr[c], errors='coerce')
ehr.to_parquet(os.path.join(OUT_DIR, f'ehr_{MODE}.parquet'), index=False)
print(f"[{MODE}] EHR per-CXR:", ehr.shape, "| coverage vital:", round(ehr.vital_heart_rate.notna().mean(),3), "| lab:", round(ehr.lab_hemoglobin.notna().mean(),3))

## 8. Dataset fusion image-level (chọn EHR_SOURCE)\n\nMuốn hybrid: chạy Cell 7 với MODE='encounter' và 'firstadm', rồi cell hybrid dưới, rồi EHR_SOURCE='hybrid'.

In [ ]:
import numpy as np, pandas as pd, os
EHR_SOURCE = 'encounter'   # 'encounter' | 'firstadm' | 'hybrid'

emb = np.load(os.path.join(OUT_DIR,'embeddings_raddino_ft_cls.npy'))
idx = pd.read_parquet(os.path.join(OUT_DIR,'embeddings_raddino_index.parquet'))
imgdf = pd.DataFrame(emb, columns=[f'img_{i}' for i in range(emb.shape[1])]); imgdf.insert(0,'dicom_id',idx['dicom_id'].values)
cm = pd.read_parquet(os.path.join(OUT_DIR,'manifest_cohort.parquet'))
lab_cols=['y_'+c for c in CHEXPERT_LABELS]; extra=[c for c in cm.columns if c=='disease_group' or c.startswith('dx_')]
meta = cm[['dicom_id','subject_id','study_id','split','view']+lab_cols+extra].drop_duplicates('dicom_id')
ehr = pd.read_parquet(os.path.join(OUT_DIR, f'ehr_{EHR_SOURCE}.parquet')).drop(columns=['subject_id','study_id'], errors='ignore')

fusion = meta.merge(imgdf, on='dicom_id', how='inner').merge(ehr, on='dicom_id', how='left')
zc=[c for c in fusion.columns if c.startswith('org_') or c.startswith('spec_')]+['n_positive_cultures','smoke','cough','ed_cough']
fusion[[c for c in zc if c in fusion.columns]] = fusion[[c for c in zc if c in fusion.columns]].fillna(0)
out = os.path.join(OUT_DIR, f'fusion_dataset_{EHR_SOURCE}')
fusion.to_parquet(out+'.parquet', index=False)
fusion[[c for c in fusion.columns if not c.startswith('img_')]].to_csv(out+'.csv', index=False)
img=[c for c in fusion.columns if c.startswith('img_')]
print(f"[{EHR_SOURCE}] fusion:", fusion.shape, "| ảnh:", fusion.dicom_id.nunique(), "| feature ảnh:", len(img), "| cột EHR/meta/nhãn:", fusion.shape[1]-len(img))

### (tuỳ chọn) tạo ehr_hybrid

In [ ]:
# (tuỳ chọn) HYBRID: encounter + lấp NaN bằng baseline firstadm. Chạy sau khi đã có ehr_encounter + ehr_firstadm
import pandas as pd, os
enc = pd.read_parquet(os.path.join(OUT_DIR,'ehr_encounter.parquet'))
fa  = pd.read_parquet(os.path.join(OUT_DIR,'ehr_firstadm.parquet'))
fill_cols=[c for c in enc.columns if c.startswith('lab_') or c.startswith('vital_')]
fa_base = fa.groupby('subject_id')[fill_cols].first()
hyb = enc.copy()
for c in fill_cols: hyb[c]=hyb[c].fillna(hyb['subject_id'].map(fa_base[c]))
hyb.to_parquet(os.path.join(OUT_DIR,'ehr_hybrid.parquet'), index=False)
print("hybrid coverage vital:", round(hyb.vital_heart_rate.notna().mean(),3), "| lab:", round(hyb.lab_hemoglobin.notna().mean(),3))

## 12. Thêm cột disease_group (bệnh nào trong 8 nhóm, tại lần nhập viện đầu) vào dataset.csv / fusion_dataset.csv\n\nChạy sau Cell 4 và Cell 11. Idempotent (chạy lại không nhân đôi cột).

In [ ]:
from google.cloud import bigquery
import pandas as pd, os
client = bigquery.Client(project=BILLING_PROJECT)

GROUPS = list(DISEASE_TITLE.keys())
dx_sel = ",\n  ".join(f"MAX(CASE WHEN {group_cond(g)} THEN 1 ELSE 0 END) AS dx_{g}" for g in GROUPS)
# disease_group = bệnh bệnh nhân TỪNG mắc qua mọi lần nằm viện (long_title, ICD-9+10), trong cohort
dxg = client.query(f"""
WITH cohort AS (SELECT DISTINCT subject_id FROM ({TARGET_ADM}))
SELECT d.subject_id,
  {dx_sel}
FROM `{H}.diagnoses_icd` d
JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
JOIN cohort c USING (subject_id)
GROUP BY d.subject_id
""").to_dataframe()

# giới hạn về cohort CXR∩EHR để số in khớp với dataset (dữ liệu ghi ra không đổi)
_cp = os.path.join(OUT_DIR,'cohort_subjects.parquet')
if os.path.exists(_cp):
    _cids = set(pd.read_parquet(_cp).subject_id)
    dxg = dxg[dxg.subject_id.isin(_cids)].reset_index(drop=True)
    print("Cohort CXR∩EHR:", len(_cids), "bệnh nhân")

dxcols = ['dx_'+g for g in GROUPS]
dxg['disease_group'] = dxg[dxcols].apply(lambda r: '|'.join(g for g,v in zip(GROUPS, r.values) if v==1), axis=1)
print("Số bệnh nhân / nhóm bệnh (long_title, ICD-9+10, mọi lần nằm viện):")
for g in GROUPS: print(f"  {g:<16}: {int(dxg['dx_'+g].sum())}")
print(dxg['disease_group'].value_counts().head(15))

def add_group(path, kind):
    if not os.path.exists(path): print("bỏ qua (chưa có):", os.path.basename(path)); return
    df = pd.read_csv(path)
    df = df.drop(columns=[c for c in dxcols+['disease_group'] if c in df.columns]).merge(dxg, on='subject_id', how='left')
    df.to_csv(path, index=False); print(f"Đã cập nhật {os.path.basename(path)} ({kind}):", df.shape)

add_group(os.path.join(OUT_DIR,'dataset.csv'),        'ảnh-level')
add_group(os.path.join(OUT_DIR,'fusion_dataset.csv'), 'bệnh nhân-level')
mc = os.path.join(OUT_DIR,'manifest_cohort.parquet')
if os.path.exists(mc):
    m = pd.read_parquet(mc)
    m = m.drop(columns=[c for c in dxcols+['disease_group'] if c in m.columns]).merge(dxg, on='subject_id', how='left')
    m.to_parquet(mc, index=False); print("Đã cập nhật manifest_cohort.parquet:", m.shape)